# PDHub Predict + Evaluate (BIOS6380)

Predict the WT and mutant structures with the ESMFold API, then compare them using four CASP-style metrics implemented in pure NumPy (Cα RMSD, TM-score, lDDT-Cα, clash score). The pair runs alongside the Mutagenesis notebook on the same variant: BRCA1 BRCT1 with the pathogenic missense p.A1708E.

What you will do:
1. Call the ESMFold API twice (WT and mutant).
2. Implement four standard structure-evaluation metrics from scratch.
3. Cross-check the structure comparison against AlphaMissense.
4. Read the per-residue confidence trace, then look at where each metric misleads.

Runs in about two minutes on a free Colab CPU. No GPU, no local models.

Every metric is re-implemented in pure NumPy below so you can read the algorithm; the full PDHub platform wraps the production tools, but those need installs we are deliberately avoiding for this teaching path.

Background: CASP14 community paper (Kryshtafovych et al. 2021, *Proteins* 89:1607) for the quality-tier thresholds.

## 0. Setup

Three small Python packages. Structure prediction is a remote API call; everything else is NumPy and matplotlib.

In [ ]:
# Quiet pip installs — < 30 s on any laptop, no GPU stack required.
%pip install -q requests pandas numpy matplotlib py3Dmol
print('Dependencies installed.')

In [ ]:
# === Target + variant input ===
# Edit these to study a different protein / variant. The worked example below is
# the BRCA1 BRCT1 domain (88 aa) with the pathogenic missense p.A1708E.

UNIPROT_ID = 'P38398'              # BRCA1 human
DOMAIN_START, DOMAIN_END = 1649, 1736   # BRCT1, 88 aa — well under the 400-aa ESMFold API cap
DOMAIN_NAME = 'BRCT1'

WT_RESIDUE = 'A'
POSITION   = 1708                  # 1-indexed in the full UniProt sequence
MT_RESIDUE = 'E'

VARIANT = f'{WT_RESIDUE}{POSITION}{MT_RESIDUE}'
TARGET  = f'{UNIPROT_ID}_{DOMAIN_NAME}'
print(f'Target: {TARGET} (residues {DOMAIN_START}–{DOMAIN_END})')
print(f'Variant: p.{VARIANT}  (within-domain index {POSITION - DOMAIN_START})')

In [ ]:
# === Fetch WT sequence from UniProt, build mutant sequence ===
import requests
from pathlib import Path

def fetch_uniprot_sequence(uniprot_id):
    r = requests.get(f'https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta', timeout=20)
    r.raise_for_status()
    return ''.join(r.text.strip().split('\n')[1:])

full_sequence = fetch_uniprot_sequence(UNIPROT_ID)
wt_domain  = full_sequence[DOMAIN_START - 1:DOMAIN_END]
rel_pos    = POSITION - DOMAIN_START         # 0-indexed inside the domain window
assert wt_domain[rel_pos] == WT_RESIDUE, f'Sanity check failed: expected {WT_RESIDUE} at position {POSITION}'
mt_domain  = wt_domain[:rel_pos] + MT_RESIDUE + wt_domain[rel_pos + 1:]

print(f'UniProt {UNIPROT_ID}: {len(full_sequence)} aa total')
print(f'Domain slice [{DOMAIN_START}-{DOMAIN_END}]: {len(wt_domain)} aa')
print(f'\nWT [±4 around {POSITION}]: ...{wt_domain[max(0, rel_pos-4):rel_pos+5]}...')
print(f'MT [±4 around {POSITION}]: ...{mt_domain[max(0, rel_pos-4):rel_pos+5]}...')

## 1. Predict the WT and mutant structures

Same endpoint and approach as the mutagenesis notebook. POST the sequence to `api.esmatlas.com/foldSequence/v1/pdb/` and get a PDB back in two or three seconds. The API caps each request at 400 residues, so we predict the BRCT1 window (88 aa) rather than full-length BRCA1.

ESMFold (Lin et al. 2023, *Science* 379) is an MSA-free single-sequence folder built on the ESM-2 protein language model.

In [ ]:
# === Predict WT and mutant via ESMFold API ===
ESMFOLD_URL = 'https://api.esmatlas.com/foldSequence/v1/pdb/'

def esmfold_predict(seq, label, timeout=180):
    print(f'  POST {label}  ({len(seq)} aa) → ESMFold API...', flush=True)
    r = requests.post(ESMFOLD_URL, data=seq, timeout=timeout,
                      headers={'Content-Type': 'text/plain'})
    r.raise_for_status()
    return r.text

wt_pdb = mt_pdb = None
try:
    wt_pdb = esmfold_predict(wt_domain, 'WT')
    mt_pdb = esmfold_predict(mt_domain, 'MT')
    Path(f'{TARGET}_WT.pdb').write_text(wt_pdb)
    Path(f'{TARGET}_{VARIANT}.pdb').write_text(mt_pdb)
    print(f'\nSaved {TARGET}_WT.pdb and {TARGET}_{VARIANT}.pdb')
except requests.exceptions.RequestException as e:
    print(f'\nESMFold API call failed: {type(e).__name__}: {e}')
    print('The ESM Atlas API may be rate-limited or temporarily down. Retry in a minute.')

## 2. 3D viewer: mutant cartoon with both side-chains

Mutant prediction in rust cartoon. Mutant side-chain at the variant in red, WT side-chain at the same position in gold from the WT model. Residues within 6 Å of the variant in grey sticks so the local pocket is visible.

In [ ]:
# === py3Dmol — WT and MT side-chains in different colours at the same site ===
# To compare side-chains visually we first Kabsch-align WT onto MT so the two
# backbones share a coordinate frame. Then we render the MT cartoon faded, the
# pocket residues as thin grey sticks, the MT side-chain in red, and the
# WT side-chain in gold at the same position.

try:
    import py3Dmol
    if wt_pdb is None or mt_pdb is None:
        print('ESMFold prediction not available — run the ESMFold cell first.')
    else:
        import numpy as np

        def _parse_ca(text):
            ca = []
            for line in text.splitlines():
                if line.startswith('ATOM') and line[12:16].strip() == 'CA':
                    ca.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            return np.array(ca)

        def _kabsch_apply(pdb_to_align, pdb_reference):
            """Kabsch-align pdb_to_align onto pdb_reference using Cα. Return new PDB text."""
            P, Q = _parse_ca(pdb_to_align), _parse_ca(pdb_reference)
            if len(P) != len(Q) or len(P) < 3:
                return pdb_to_align
            Pc, Qc = P.mean(0), Q.mean(0)
            H = (P - Pc).T @ (Q - Qc)
            U, _, Vt = np.linalg.svd(H)
            d = np.sign(np.linalg.det(Vt.T @ U.T))
            R = Vt.T @ np.diag([1, 1, d]) @ U.T
            t = Qc - R @ Pc
            out = []
            for line in pdb_to_align.splitlines():
                if line.startswith('ATOM') and len(line) >= 54:
                    xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
                    n = R @ xyz + t
                    out.append(line[:30] + f'{n[0]:8.3f}{n[1]:8.3f}{n[2]:8.3f}' + line[54:])
                else:
                    out.append(line)
            return '\n'.join(out)

        wt_pdb_aligned = _kabsch_apply(wt_pdb, mt_pdb)
        resi_in_pdb = str(rel_pos + 1)
        view = py3Dmol.view(width=720, height=440)

        # MT model: faded cartoon + local pocket sticks + red mutant side-chain
        view.addModel(mt_pdb, 'pdb')
        view.setStyle({'model': 0}, {'cartoon': {'color': '#8B3018', 'opacity': 0.25}})
        view.addStyle({'model': 0, 'within': {'distance': 6.0, 'sel': {'resi': resi_in_pdb}}},
                      {'stick': {'color': '#bcbcbc', 'radius': 0.18, 'opacity': 0.85}})
        view.addStyle({'model': 0, 'resi': resi_in_pdb},
                      {'stick': {'color': 'red', 'radius': 0.45}})

        # WT model (aligned): hidden cartoon, WT side-chain at the residue in gold
        view.addModel(wt_pdb_aligned, 'pdb')
        view.setStyle({'model': 1}, {})
        view.addStyle({'model': 1, 'resi': resi_in_pdb},
                      {'stick': {'color': 'gold', 'radius': 0.35}})

        view.zoomTo({'model': 0, 'resi': resi_in_pdb})
        view.zoom(0.35)
        view.show()
        print(f'Tight residue-level view of {VARIANT}.')
        print(f'Red sticks    = MT {MT_RESIDUE}{POSITION} side-chain.')
        print(f'Gold sticks   = WT {WT_RESIDUE}{POSITION} side-chain (Kabsch-aligned onto MT frame).')
        print(f'Grey sticks   = pocket residues within 6 Å of the variant.')
        print(f'Faded cartoon = MT backbone for spatial context.')
except ImportError:
    print('py3Dmol not installed — run the setup cell.')
except Exception as e:
    print(f'3D viewer skipped: {e}')

## 3. Evaluate: four metrics from scratch

We compare the mutant prediction to the WT prediction with four standard structure-evaluation metrics. Each is re-implemented in pure NumPy below so the algorithm is on the page.

- Cα RMSD after Kabsch superposition (Kabsch 1976).
- TM-score (Zhang & Skolnick 2004), length-normalised fold similarity.
- lDDT-Cα (Mariani 2013), local distance preservation, superposition-free.
- A simplified clash score: heavy-atom pairs in steric conflict, per 1000 atoms.

In [ ]:
# === PDB parsing helpers ===
import numpy as np

def parse_pdb_atoms(pdb_text, want_atoms=None):
    """
    Parse ATOM records. Returns (resnums, atomnames, coords, bfactors).
    `want_atoms`: e.g. {'CA'} to keep only Cα.
    """
    resnums, atomnames, coords, bfactors = [], [], [], []
    seen = set()
    for line in pdb_text.splitlines():
        if not line.startswith('ATOM'):
            continue
        atom = line[12:16].strip()
        if want_atoms is not None and atom not in want_atoms:
            continue
        altloc = line[16].strip()
        if altloc and altloc != 'A':
            continue
        try:
            resnum = int(line[22:26])
        except ValueError:
            continue
        key = (resnum, atom)
        if key in seen:
            continue
        seen.add(key)
        resnums.append(resnum)
        atomnames.append(atom)
        coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
        try:
            bfactors.append(float(line[60:66]))
        except ValueError:
            bfactors.append(0.0)
    return (np.array(resnums), np.array(atomnames, dtype=object),
            np.array(coords), np.array(bfactors))

wt_resnums_ca, _, wt_ca, wt_b = parse_pdb_atoms(wt_pdb, want_atoms={'CA'})
mt_resnums_ca, _, mt_ca, mt_b = parse_pdb_atoms(mt_pdb, want_atoms={'CA'})
wt_resnums_h,  _, wt_heavy, _ = parse_pdb_atoms(wt_pdb)
mt_resnums_h,  _, mt_heavy, _ = parse_pdb_atoms(mt_pdb)

# ESMFold pLDDT comes out in 0–1 from the API; rescale to the standard 0–100.
wt_plddt = wt_b * 100.0
mt_plddt = mt_b * 100.0

print(f'WT:  {len(wt_ca):>3d} Cα  ({len(wt_heavy):>4d} heavy atoms)  mean pLDDT = {wt_plddt.mean():.1f}')
print(f'MT:  {len(mt_ca):>3d} Cα  ({len(mt_heavy):>4d} heavy atoms)  mean pLDDT = {mt_plddt.mean():.1f}')
assert (wt_resnums_ca == mt_resnums_ca).all(), 'WT and MT predictions have mismatched residue numbering'
common_resnums = wt_resnums_ca

In [ ]:
# === Metric 1 — Cα RMSD after Kabsch alignment ===
# Kabsch (1976): optimal rotation that minimises RMSD between two point sets.

def kabsch_align(P, Q):
    Pc, Qc = P.mean(0), Q.mean(0)
    H = (P - Pc).T @ (Q - Qc)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    return R, Qc - R @ Pc

def kabsch_rmsd(P, Q):
    R, t = kabsch_align(P, Q)
    P_fit = (P @ R.T) + t
    return float(np.sqrt(np.mean(np.sum((P_fit - Q) ** 2, axis=1)))), P_fit

rmsd_ca, mt_ca_fit = kabsch_rmsd(mt_ca, wt_ca)
per_residue_dev = np.linalg.norm(mt_ca_fit - wt_ca, axis=1)
print(f'Cα RMSD (Kabsch-aligned, WT vs MT)  = {rmsd_ca:.3f} Å')
print(f'Worst-residue deviation             = {per_residue_dev.max():.3f} Å '
      f'at residue {common_resnums[per_residue_dev.argmax()]}')

In [ ]:
# === Metric 2 — TM-score (Zhang & Skolnick 2004) ===
# TM-score = (1/L) · Σ_i 1 / (1 + (d_i / d0)^2)
# where d0 = 1.24 · (L - 15)^(1/3) - 1.8  for L ≥ 22, else d0 = 0.5.
#
# Interpretation (Xu & Zhang 2010):
#   TM < 0.17  random unrelated structures
#   0.17–0.50  partial similarity
#   > 0.50     same fold
#   > 0.85     near-identical

def tm_score_from_distances(distances, L):
    d0 = 1.24 * (L - 15) ** (1/3) - 1.8 if L >= 22 else 0.5
    return float(np.mean(1.0 / (1.0 + (distances / d0) ** 2))), d0

tm, d0 = tm_score_from_distances(per_residue_dev, L=len(wt_ca))
tier = ('near-identical' if tm > 0.85 else 'same fold' if tm > 0.5
        else 'partial similarity' if tm > 0.17 else 'unrelated')
print(f'TM-score (WT vs MT, Kabsch-aligned) = {tm:.3f}   (d0 = {d0:.2f} Å, L = {len(wt_ca)})')
print(f'Quality tier: {tier}')

In [ ]:
# === Metric 3 — lDDT-Cα (Mariani 2013) ===
# Local Distance Difference Test: superposition-free.
# For each Cα–Cα pair within INCLUSION_RADIUS Å in the WT (treated as reference),
# check whether the mutant preserves that distance within {0.5, 1, 2, 4} Å.
# lDDT_residue = mean fraction preserved across thresholds for all pairs involving
# that residue. lDDT_global = mean over residues.

INCLUSION_RADIUS = 15.0
THRESHOLDS = (0.5, 1.0, 2.0, 4.0)
SEQ_SEPARATION = 0

def lddt_ca(model_ca, ref_ca, inclusion_radius=INCLUSION_RADIUS,
            thresholds=THRESHOLDS, seq_separation=SEQ_SEPARATION):
    n = len(ref_ca)
    assert len(model_ca) == n, 'lDDT requires 1:1 residue correspondence'
    ref_d = np.linalg.norm(ref_ca[:, None] - ref_ca[None, :], axis=-1)
    mod_d = np.linalg.norm(model_ca[:, None] - model_ca[None, :], axis=-1)
    i_idx, j_idx = np.indices((n, n))
    valid = (ref_d < inclusion_radius) & (ref_d > 0) & (np.abs(i_idx - j_idx) > seq_separation)
    abs_diff = np.abs(mod_d - ref_d)
    per_res = np.zeros(n)
    for i in range(n):
        mask = valid[i]
        if mask.sum() == 0:
            per_res[i] = np.nan
            continue
        per_res[i] = np.mean([(abs_diff[i][mask] < t).mean() for t in thresholds])
    return per_res, float(np.nanmean(per_res))

per_res_lddt, global_lddt = lddt_ca(mt_ca, wt_ca)
print(f'lDDT-Cα global (WT vs MT) = {global_lddt:.3f}   '
      f'(inclusion radius {INCLUSION_RADIUS} Å, thresholds {THRESHOLDS} Å)')
worst = np.nanargmin(per_res_lddt)
print(f'Worst lDDT-Cα residue     = {per_res_lddt[worst]:.3f}  at residue {common_resnums[worst]}')

In [ ]:
# === Metric 4 — Clash score (simplified) ===
# Count heavy-atom pairs closer than CUTOFF Å, excluding atoms within ±SEP_AA
# residues of each other (so we don't flag covalent bonds and near-neighbour
# contacts). Normalised per 1000 atoms.

CLASH_CUTOFF = 2.0
SEP_AA       = 2

def clash_score(coords, resnums, cutoff=CLASH_CUTOFF, sep_aa=SEP_AA):
    if len(coords) < 2:
        return 0.0, 0
    d = np.linalg.norm(coords[:, None] - coords[None, :], axis=-1)
    res_diff = np.abs(resnums[:, None] - resnums[None, :])
    keep = (d < cutoff) & (d > 0) & (res_diff > sep_aa)
    clashes = int(keep.sum() // 2)
    norm = clashes / (len(coords) / 1000.0)
    return float(norm), clashes

wt_clash_norm, wt_clashes = clash_score(wt_heavy, wt_resnums_h)
mt_clash_norm, mt_clashes = clash_score(mt_heavy, mt_resnums_h)
print(f'WT clash score: {wt_clashes:>3d} pairs ({wt_clash_norm:.2f} per 1000 atoms)')
print(f'MT clash score: {mt_clashes:>3d} pairs ({mt_clash_norm:.2f} per 1000 atoms)')

In [ ]:
# === Summary at the variant residue (WT vs MT) ===
import pandas as pd
at_residue = pd.DataFrame([
    {'metric': f'WT pLDDT @ residue {POSITION}',
     'value': round(float(wt_plddt[rel_pos]), 1), 'units': '0-100'},
    {'metric': f'MT pLDDT @ residue {POSITION}',
     'value': round(float(mt_plddt[rel_pos]), 1), 'units': '0-100'},
    {'metric': 'ΔpLDDT (MT − WT) @ variant',
     'value': round(float(mt_plddt[rel_pos] - wt_plddt[rel_pos]), 2), 'units': '0-100 delta'},
    {'metric': f'Cα displacement @ {POSITION}',
     'value': round(float(per_residue_dev[rel_pos]), 3), 'units': 'Å'},
])
print(at_residue.to_string(index=False))
# Global metrics (RMSD, TM, lDDT) are printed individually in the four metric cells above.

## 4. Cross-check with AlphaMissense

The four metrics above compare predicted structures. AlphaMissense (Cheng et al. 2023, *Science* 381) scores variant pathogenicity directly from sequence; same ESM-2 backbone, trained for variant effect prediction.

If structure says near-identical and AlphaMissense says likely pathogenic, that disagreement is where the variant signal lives.

In [ ]:
# === AlphaMissense lookup ===
# DeepMind released pre-computed scores for all ~ 71 M canonical human missense
# variants. The full table is ~ 1 GB; here we ship only the worked-example entry.
# For other variants, download the public table from
# https://storage.googleapis.com/dm_alphamissense/ and replace the lookup.

ALPHAMISSENSE_CACHE = {
    ('P38398', 'A1708E'): {'score': 0.94, 'class': 'likely_pathogenic', 'threshold': 0.564},
}

def fetch_alphamissense(uniprot_id, variant):
    return ALPHAMISSENSE_CACHE.get((uniprot_id, variant))

am = fetch_alphamissense(UNIPROT_ID, VARIANT)
if am is None:
    print(f'AlphaMissense entry for {UNIPROT_ID} · {VARIANT} not cached.')
    print('Download the public lookup table to extend (see comment above).')
else:
    print(f'AlphaMissense score = {am["score"]:.2f}  (pathogenic threshold ≥ {am["threshold"]:.3f})')
    print(f'AlphaMissense class = {am["class"].upper()}')

### 4.5. WT vs mutant side-chain

So far we have compared whole structures. The variant is a single side-chain swap. The table below summarises the physicochemical difference (Ala vs Glu) alongside the per-residue pLDDT from both predictions.

In [ ]:
# === Master WT vs MT comparison — residue-level only ===
# Pure-Python physicochemical lookup (Kyte–Doolittle hydropathy + Zamyatnin volume).
AA_PROPS = {
    'A': ('Ala',  0.0,  +1.8,  88.6,  'hydrophobic'),
    'R': ('Arg', +1.0,  -4.5, 173.4,  'positively charged'),
    'N': ('Asn',  0.0,  -3.5, 114.1,  'polar'),
    'D': ('Asp', -1.0,  -3.5, 111.1,  'negatively charged'),
    'C': ('Cys',  0.0,  +2.5, 108.5,  'sulfur-containing'),
    'E': ('Glu', -1.0,  -3.5, 138.4,  'negatively charged'),
    'Q': ('Gln',  0.0,  -3.5, 143.8,  'polar'),
    'G': ('Gly',  0.0,  -0.4,  60.1,  'small'),
    'H': ('His', +0.5,  -3.2, 153.2,  'positively charged'),
    'I': ('Ile',  0.0,  +4.5, 166.7,  'hydrophobic'),
    'L': ('Leu',  0.0,  +3.8, 166.7,  'hydrophobic'),
    'K': ('Lys', +1.0,  -3.9, 168.6,  'positively charged'),
    'M': ('Met',  0.0,  +1.9, 162.9,  'sulfur-containing'),
    'F': ('Phe',  0.0,  +2.8, 189.9,  'aromatic'),
    'P': ('Pro',  0.0,  -1.6, 112.7,  'helix-breaker'),
    'S': ('Ser',  0.0,  -0.8,  89.0,  'polar'),
    'T': ('Thr',  0.0,  -0.7, 116.1,  'polar'),
    'W': ('Trp',  0.0,  -0.9, 227.8,  'aromatic'),
    'Y': ('Tyr',  0.0,  -1.3, 193.6,  'aromatic'),
    'V': ('Val',  0.0,  +4.2, 140.0,  'hydrophobic'),
}

import pandas as pd
wt3, wt_q, wt_h, wt_v, wt_c = AA_PROPS[WT_RESIDUE]
mt3, mt_q, mt_h, mt_v, mt_c = AA_PROPS[MT_RESIDUE]

def fmt(x, n=2):
    return f'{x:.{n}f}' if isinstance(x, float) else str(x)

rows = [
    ('Residue identity',
     f'{wt3} ({WT_RESIDUE})', f'{mt3} ({MT_RESIDUE})',
     f'{WT_RESIDUE} → {MT_RESIDUE}',
     f'{wt_c} → {mt_c}'),
    ('Charge (e)', fmt(wt_q, 1), fmt(mt_q, 1), fmt(mt_q - wt_q, 1),
     'net charge change at the position'),
    ('Hydropathy (Kyte–Doolittle)', fmt(wt_h, 1), fmt(mt_h, 1), fmt(mt_h - wt_h, 1),
     'negative = more hydrophilic; |Δ| > 4 is a large swing'),
    ('Side-chain volume (Å³)', fmt(wt_v, 1), fmt(mt_v, 1), fmt(mt_v - wt_v, 1),
     '+50 Å³ swap can strain a packed pocket'),
    (f'pLDDT @ residue {POSITION}',
     fmt(float(wt_plddt[rel_pos]), 1), fmt(float(mt_plddt[rel_pos]), 1),
     fmt(float(mt_plddt[rel_pos] - wt_plddt[rel_pos]), 2),
     'structure-model self-confidence at the residue'),
    (f'Cα displacement @ {POSITION}', '—', '—',
     f'{float(per_residue_dev[rel_pos]):.3f} Å',
     'sub-Ångström = structure-model SILENT at the residue'),
    ('AlphaMissense score (variant)', '—', '—',
     f'{am["score"]:.2f} (≥ {am["threshold"]:.3f})' if am else 'n/a',
     (am['class'].replace('_', ' ').upper() + ' — sequence model fires at this residue')
       if am else 'no cached entry'),
]
master = pd.DataFrame(rows, columns=['metric', 'WT', 'MT', 'Δ / verdict', 'interpretation'])
print('=== WT vs MT — at the variant residue ===')
print(master.to_string(index=False))

# Variant-residue delta exposed for the verdict cells
delta_plddt_residue = float(mt_plddt[rel_pos] - wt_plddt[rel_pos])


In [ ]:
# === Master comparison figure — physicochemical + structure + pathogenicity ===
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(13, 4.0), gridspec_kw={'width_ratios': [2.2, 1.6, 1.4]})

# Panel 1: physicochemical bar pairs (charge / hydropathy / volume)
props   = ['Charge\n(e)', 'Hydropathy\n(Kyte–Doolittle)', 'Volume\n(Å³)']
wt_vals = [AA_PROPS[WT_RESIDUE][1], AA_PROPS[WT_RESIDUE][2], AA_PROPS[WT_RESIDUE][3]]
mt_vals = [AA_PROPS[MT_RESIDUE][1], AA_PROPS[MT_RESIDUE][2], AA_PROPS[MT_RESIDUE][3]]
# Volume needs its own axis (different scale); plot as fractions for shape, annotate values
ax = axes[0]
x = np.arange(3); width = 0.36
# Normalise each property to [-1, +1] band where possible
def norm(v, lo, hi):
    return 2 * (v - lo) / (hi - lo) - 1 if hi > lo else 0
norm_wt = [norm(wt_vals[0], -2, 2), norm(wt_vals[1], -5, 5), norm(wt_vals[2], 0, 250)]
norm_mt = [norm(mt_vals[0], -2, 2), norm(mt_vals[1], -5, 5), norm(mt_vals[2], 0, 250)]
ax.bar(x - width/2, norm_wt, width, color='#4FB3BF', label=f'WT ({WT_RESIDUE})', edgecolor='none')
ax.bar(x + width/2, norm_mt, width, color='#8B3018', label=f'MT ({MT_RESIDUE})', edgecolor='none')
ax.axhline(0, color='#666', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(props, fontsize=9)
ax.set_ylabel('normalised value (each prop has its own scale)')
ax.set_title('Side-chain physicochemical properties', fontsize=10, color='#1F2A47')
ax.legend(loc='lower left', fontsize=8)
# Annotate raw values
for i, (w, m) in enumerate(zip(wt_vals, mt_vals)):
    ax.text(i - width/2, norm_wt[i] + 0.06, f'{w}', ha='center', fontsize=8, color='#1F2A47')
    ax.text(i + width/2, norm_mt[i] + 0.06, f'{m}', ha='center', fontsize=8, color='#8B3018')
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.grid(axis='y', alpha=0.25); ax.set_ylim(-1.2, 1.2)

# Panel 2: structure metrics at the variant residue
ax = axes[1]
labels = [f'pLDDT @ {POSITION}', f'Cα displacement @ {POSITION}\n(Å)']
wt_struct = [float(wt_plddt[rel_pos]), 0.0]
mt_struct = [float(mt_plddt[rel_pos]), float(per_residue_dev[rel_pos])]
xs = np.arange(2)
ax.bar(xs - width/2, wt_struct, width, color='#4FB3BF', edgecolor='none', label='WT')
ax.bar(xs + width/2, mt_struct, width, color='#8B3018', edgecolor='none', label='MT')
ax.set_xticks(xs); ax.set_xticklabels(labels, fontsize=9)
ax.set_title('Structure metrics @ variant residue', fontsize=10, color='#1F2A47')
ax.legend(loc='upper right', fontsize=8)
for i, (w, m) in enumerate(zip(wt_struct, mt_struct)):
    ax.text(i - width/2, w + 1.5, f'{w:.2f}', ha='center', fontsize=8, color='#1F2A47')
    ax.text(i + width/2, m + 1.5, f'{m:.2f}', ha='center', fontsize=8, color='#8B3018')
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.grid(axis='y', alpha=0.25)

# Panel 3: AlphaMissense score (mutant-only, with pathogenic threshold)
ax = axes[2]
if am is not None:
    ax.bar([0], [am['score']], color='#8B3018', width=0.5, edgecolor='none')
    ax.axhline(am['threshold'], color='#1F2A47', ls='--', lw=1.2)
    ax.text(0, am['score'] + 0.03, f'{am["score"]:.2f}', ha='center', fontsize=10,
            color='#1F2A47', fontweight='bold')
    ax.text(0.32, am['threshold'] + 0.01, f'threshold {am["threshold"]:.3f}',
            fontsize=8, color='#1F2A47')
    ax.set_ylim(0, 1.15)
    ax.set_xticks([0]); ax.set_xticklabels([am['class'].replace('_', ' ').upper()],
                                            fontsize=9, rotation=20, ha='right')
    ax.set_ylabel('AlphaMissense (0 = benign · 1 = pathogenic)')
else:
    ax.text(0.5, 0.5, 'AlphaMissense\nnot cached', ha='center', va='center', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Pathogenicity (sequence model)', fontsize=10, color='#1F2A47')
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.grid(axis='y', alpha=0.25)

plt.tight_layout(); plt.show()

## 6. Verdict

Combine the four structure metrics with the AlphaMissense cross-check. The interesting case is when the two disagree.

In [ ]:
# === Combined verdict ===
def structure_verdict(rmsd, tm, lddt, dplddt):
    if tm > 0.85 and lddt > 0.85 and rmsd < 2.0 and abs(dplddt) < 5:
        return 'NEAR-IDENTICAL — structure-comparison metrics SILENT on this variant'
    if tm > 0.5 and lddt > 0.6:
        return 'SAME FOLD — minor deviations, weak signal'
    if tm > 0.17:
        return 'PARTIAL SIMILARITY — strong structural signal'
    return 'UNRELATED — model collapsed (very rare for a single missense)'

def alphamissense_verdict(am):
    if am is None:
        return 'NOT CACHED'
    return am['class'].replace('_', ' ').upper()

sv = structure_verdict(rmsd_ca, tm, global_lddt, delta_plddt_residue)
av = alphamissense_verdict(am)

print('=' * 60)
print('  STRUCTURE-COMPARISON VERDICT')
print(f'  {sv}')
print()
print('  ALPHAMISSENSE VERDICT')
print(f'  {av}'
      + (f'   (score {am["score"]:.2f}, threshold {am["threshold"]:.3f})' if am else ''))
print('=' * 60)
print(f'  Cα RMSD              = {rmsd_ca:.3f} Å')
print(f'  TM-score (WT vs MT)  = {tm:.3f}')
print(f'  lDDT-Cα (WT vs MT)   = {global_lddt:.3f}')
print(f'  ΔpLDDT at residue    = {delta_plddt_residue:+.2f}')
print(f'  Clash WT / MT        = {wt_clash_norm:.2f} / {mt_clash_norm:.2f} per 1000 atoms')
if am is not None:
    print(f'  AlphaMissense score  = {am["score"]:.2f}')
print()
print('When structure metrics say NEAR-IDENTICAL but AlphaMissense says LIKELY PATHOGENIC,')
print('the variant signal is in the sequence model, not in the predicted geometry. The')
print('mutagenesis notebook adds conservation and functional-assay evidence in the')
print('ACMG/AMP framework.')

## Residue chemistry: what the A → E swap actually does

The mutation replaces an alanine (small, non-polar, methyl side-chain) with a glutamate (longer side-chain, negatively charged carboxylate at physiological pH). The four properties shown in the master table all change at once:

- **Charge** 0 → −1. A new fixed negative charge appears at residue 1708. In a buried site this charge has no nearby counter-ion, which is energetically expensive.
- **Hydropathy** +1.8 → −3.5 (Kyte–Doolittle). A 5.3-unit swing toward hydrophilic. The residue used to fit comfortably in a hydrophobic environment; the mutant prefers solvent.
- **Volume** 88.6 → 138.4 Å³. A 50 Å³ larger side-chain. In a packed pocket, the extra volume strains the local fold even when the backbone holds together.
- **Polarity class** hydrophobic → negatively charged. The chemical character flips.

In the BRCT phospho-peptide binding pocket, introducing a negative charge near the binding face is exactly the kind of chemistry change that disrupts function. The backbone fold is robust enough to absorb the substitution, which is why ESMFold sees nothing. AlphaMissense (sequence-based) and conservation (evolutionary) both fire loudly because they read the variant signal where it actually lives — in the sequence and in the evolutionary record, not in the predicted backbone.

### Why the comparison table shows only residue-level metrics

The four CASP-style metrics (Cα RMSD, TM-score, lDDT-Cα, clash score) compute over the whole domain. For a single missense, averaging across 88 residues dilutes the local signal — any per-residue change gets divided by 88. The comparison table leads with residue-level rows (ΔpLDDT at the variant, Cα displacement at the variant, AlphaMissense for the variant) so the signal is where you can see it. The four global metrics still print individually in the algorithm cells above; the table just doesn't lead with them.

AlphaMissense is variant-specific by construction. It reads the (UniProt, variant) pair and returns a probabilistic pathogenicity score. It belongs in the residue-level table, not in the global-structure block.

### What stays at the global level, and why

The pedagogical content of the notebook is in the four CASP metric cells (Cα RMSD, TM-score, lDDT-Cα, clash score). Those cells exist so a student can read the algorithm in pure NumPy and understand how each metric is computed. The summary tables consume those values, but they are not the place to learn the algorithms — the algorithm cells are. Hence: algorithm cells keep their global prints; summary tables stay residue-focused.

## 7. Where each tool misleads

| Tool | What it tells you | What it misses |
|---|---|---|
| Cα RMSD | average backbone deviation after global alignment | one bad loop drags the mean; side-chain rearrangement invisible |
| TM-score | fold-level similarity, length-normalised | insensitive to local errors; saturates above 0.85 |
| lDDT-Cα | local distance preservation, superposition-free | Cα-only; side-chain misplacement invisible |
| pLDDT | model self-confidence | can be confidently wrong; trained on backbone, not function |
| Clash score | steric impossibility | this simplified version ignores van der Waals radii and bonding graphs |
| AlphaMissense | pre-trained pathogenicity | novel domains the training set never saw; non-canonical isoforms |

### Questions to work through

1. Every structure metric says NEAR-IDENTICAL, AlphaMissense says LIKELY PATHOGENIC. How do you reconcile?
2. The ΔpLDDT at the variant residue is tiny. Under what conditions would you expect ΔpLDDT to actually flag a deleterious missense?
3. Cα-lDDT and all-atom lDDT can disagree. Construct an example where Cα-lDDT looks fine but all-atom lDDT collapses.
4. Run the pipeline with a benign missense (e.g. a conservative substitution in a flexible loop). Does the metric stack still look the same? What does that tell you?

---

## References

- Lin et al. (2023) *Science* 379 — ESMFold.
- Mariani et al. (2013) *Bioinformatics* 29:2722 — lDDT.
- Zhang and Skolnick (2004) *Proteins* 57:702 — TM-score.
- Kabsch (1976) *Acta Crystallogr* A32:922.
- Kryshtafovych et al. (2021) *Proteins* 89:1607 — CASP14.
- Cheng et al. (2023) *Science* 381 — AlphaMissense.
- Buel and Walters (2022) *Nat Struct Mol Biol* 29:1–2.

### Variant-specific references (BRCA1 A1708E)

- Williams et al. (2003) *Nat Struct Mol Biol* — BRCA1 BRCT phospho-peptide pocket structure.
- Findlay et al. (2018) *Nature* — saturation genome editing of BRCA1, A1708E loss-of-function.
- Richards et al. (2015) *Genet Med* 17:405 — ACMG/AMP combining rules.

Repo: [github.com/recep2244/pdhub](https://github.com/recep2244/pdhub), MIT.

Module: BIOS6380. Notebook: PDHub_Predict_Evaluate_BIOS6380.ipynb v1.3.